# skyflat

> An example of computing a skyflat from a set of observations.

::: {.callout-warning}
This notebook takes 2-3 hours to create a skyflat from all of the Q1 data.
:::

In [ ]:
from nicl.euclid.utilities import default_data_path
from nicl.euclid.xarray import (
    create_zarr_ref_from_fits,
    load_zarr_ref_from_file,
    open_zarr_ref_as_dataset,
    write_da_to_fits,
)

In [ ]:
path = default_data_path("Q1_R1_processed_v0.2", "persistence")

In [ ]:
fns = [str(p) for p in (path / "NIR").glob("**/*.fits")]

In [ ]:
%%time
zarrfn = path / "zarr.test.ref"
if not zarrfn.exists():
    ref = create_zarr_ref_from_fits(fns, return_wcs=False, outfn=zarrfn)
else:
    ref = load_zarr_ref_from_file(zarrfn)

In [ ]:
%%time
ds = open_zarr_ref_as_dataset(ref)

In [ ]:
skyflat = ds["SCI"].median(dim=["observation_id", "dither"], keep_attrs=True)

In [ ]:
_, wcs_out = create_zarr_ref_from_fits(fns[0], return_wcs=True)
wcs_out = wcs_out.squeeze(drop=True)
wcs_out["CRVAL1"] = 0
wcs_out["CRVAL2"] = 0

In [ ]:
%%time
hdul = {}
for filt in skyflat.filter.data:
    hdul[filt] = write_da_to_fits(
        skyflat.sel(filter=filt),
        path / f"../skyflat-{filt}.fits",
        da_wcs=wcs_out,
        overwrite=True,
    )